# Safe-Guard Dataset Inventory

Goal: create a reproducible inventory of the hosted dataset without committing raw data files. Keep the provided test split held out for final evaluation.


## 4. Save only summary outputs



## 1. Set up the notebook

Install `datasets`, `huggingface_hub`, and `pandas`. Load `xTRam1/safe-guard-prompt-injection`, record its revision hash, and use that revision when loading the data.

In [ ]:
!pip -q install datasets huggingface_hub pandas

from datasets import load_dataset
from huggingface_hub import HfApi
import pandas as pd

DATASET_ID = "xTRam1/safe-guard-prompt-injection"
REVISION = HfApi().dataset_info(DATASET_ID).sha
print("Dataset:", DATASET_ID)
print("Revision:", REVISION)

ds = load_dataset(DATASET_ID, revision=REVISION)
print(ds)

Dataset: xTRam1/safe-guard-prompt-injection
Revision: a3a877d608f37b7d20d9945671902df895ecdb46


README.md:   0%|          | 0.00/2.76k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  497kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8236 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2060 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8236
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2060
    })
})


## 2. Inventory each split

For train and test, record the file/split name, row count, columns and types, null or empty text, distinct label values, class counts/proportions, prompt lengths, and exact duplicates. Do not silently coerce malformed values.

In [ ]:
EXPECTED_COLUMNS = {"text", "label"}
EXPECTED_LABELS = {0, 1}
summary_rows, quality_rows = [], []

for split_name, split_data in ds.items():
    df = split_data.to_pandas()
    missing_columns = sorted(EXPECTED_COLUMNS - set(df.columns))
    unexpected_labels = sorted(set(df["label"].dropna().unique()) - EXPECTED_LABELS)
    text_lengths = df["text"].fillna("").astype(str).str.len()

    for label, count in df["label"].value_counts(dropna=False).sort_index().items():
        summary_rows.append({"split": split_name, "label": label, "count": int(count),
                             "proportion": round(count / len(df), 4)})

    quality_rows.append({
        "split": split_name, "rows": len(df), "columns": list(df.columns),
        "missing_expected_columns": missing_columns,
        "null_text_rows": int(df["text"].isna().sum()),
        "empty_or_whitespace_text_rows": int(df["text"].fillna("").astype(str).str.strip().eq("").sum()),
        "unexpected_label_values": unexpected_labels,
        "duplicate_rows_beyond_first": int(df.duplicated(subset=["text"]).sum()),
        "prompt_length_min": int(text_lengths.min()),
        "prompt_length_median": float(text_lengths.median()),
        "prompt_length_mean": round(float(text_lengths.mean()), 2),
        "prompt_length_max": int(text_lengths.max()),
    })

class_balance = pd.DataFrame(summary_rows)
quality_summary = pd.DataFrame(quality_rows)
display(class_balance)
display(quality_summary)

,split,label,count,proportion
0,train,0,5740,0.6969
1,train,1,2496,0.3031
2,test,0,1410,0.6845
3,test,1,650,0.3155


,split,rows,columns,missing_expected_columns,null_text_rows,empty_or_whitespace_text_rows,unexpected_label_values,duplicate_rows_beyond_first,prompt_length_min,prompt_length_median,prompt_length_mean,prompt_length_max
0,train,8236,"[text, label]",[],0,0,[],113,11,140.0,385.26,12809
1,test,2060,"[text, label]",[],0,0,[],11,14,136.0,367.62,9033


## 3. Check exact train/test overlap

In [ ]:
train_text = set(ds["train"]["text"])
test_text = set(ds["test"]["text"])
exact_overlap = train_text.intersection(test_text)

print("Exact train/test prompt overlap:", len(exact_overlap))
print("Unique train prompts:", len(train_text))
print("Unique test prompts:", len(test_text))

Exact train/test prompt overlap: 35
Unique train prompts: 8123
Unique test prompts: 2049



## 4. Save only summary outputs

In [ ]:
class_balance.to_csv("safe_guard_class_balance.csv", index=False)
quality_summary.to_csv("safe_guard_inventory_summary.csv", index=False)

# Issue #23 — Safe-Guard Dataset Inventory Report

## Dataset and reproducibility

The notebook loads `xTRam1/safe-guard-prompt-injection` and records the Hugging Face revision hash in `REVISION`. The dataset is a binary prompt-classification task with the expected `text` and `label` fields, where label `0` denotes safe/benign prompts and label `1` denotes prompt-injection prompts. The provided test split remains separate from the training data.

## Split composition

| Split | Rows | Safe / label 0 | Injection / label 1 |
|---|---:|---:|---:|
| Train | 8,236 | 5,740 (69.69%) | 2,496 (30.31%) |
| Test | 2,060 | 1,410 (68.45%) | 650 (31.55%) |

The class balance is similar across the two splits. Both contain the expected two columns, with no missing expected fields and no unexpected label values.

## Text quality and prompt length

No null, empty, or whitespace-only prompts were found in either split.

| Split | Median length | Mean length | Minimum | Maximum |
|---|---:|---:|---:|---:|
| Train | 140 characters | 385.26 | 11 | 12,809 |
| Test | 136 characters | 367.62 | 14 | 9,033 |

The mean is substantially larger than the median in both splits, indicating a right-skewed prompt-length distribution caused by a smaller number of very long prompts. Later modeling should avoid assuming that a typical prompt has the mean length.

## Duplicates and split overlap

The train split contains 113 duplicate rows beyond the first occurrence; the test split contains 11. There are 35 exact prompt strings shared by train and test. After deduplication, there are 8,123 unique train prompts and 2,049 unique test prompts.

These exact overlaps are a potential leakage risk: a model could receive identical text during training and evaluation. This inventory does not establish whether all overlaps have conflicting labels or materially inflate evaluation metrics. The next step is to inspect the shared prompts, confirm label consistency, and decide whether a deduplicated evaluation protocol is needed before reporting model performance.

## Scope and limitations

This hosted dataset exposes text and binary labels only. It does not natively provide language, topic, source, or attack-family fields. Therefore, this inventory cannot support claims about performance, bias, or coverage across those groups without explicitly documented and reproducible derived annotations.

## Follow-up

- Compare schema and label findings with Zeynep's loader-validation work (#26).
- Link the dataset provenance and licensing notes from Jacqueline's work (#24).
- Share the duplicate and train/test-overlap findings with Ahmed's leakage audit (#28).
